In [26]:
import pandas as pd
import numpy as np

In [27]:
def rank_tfs(Xi, source_cell, target_cell, top_n=3):
    """
    Rank transcription factors by predictivity gap.
    
    Parameters
    ----------
    Xi          : DataFrame (genes x cell types)
    source_cell : str - name of starting cell type
    target_cell : str - name of desired cell type
    top_n       : int - number of top TFs to return

    Returns
    -------
    DataFrame with TFs ranked by gap score
    """
    if source_cell not in Xi.columns:
        raise ValueError(f"Source '{source_cell}' not found. Available: {Xi.columns.tolist()}")
    if target_cell not in Xi.columns:
        raise ValueError(f"Target '{target_cell}' not found. Available: {Xi.columns.tolist()}")

    xi_target = Xi[target_cell]
    xi_others = Xi.drop(columns=[target_cell])
    xi_source = Xi[source_cell]

    gap = xi_target - xi_others.max(axis=1)
    difference = xi_target - xi_source
    direction = difference.apply(lambda x: 'overexpress' if x > 0 else 'knockdown')

    results = pd.DataFrame({
        'gap_score': gap,
        'source_expression': xi_source,
        'target_expression': xi_target,
        'direction': direction
    }).sort_values('gap_score', ascending=False)

    return results.head(top_n)

In [28]:
def softmax(z):
    """
    Convert a vector of scores into a probability distribution.
    Subtracts max first for numerical stability.
    """
    z = z - np.max(z)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z)

In [29]:
def dx_dt(x, Xi, beta, w, v):
    """
    dx/dt = v + Xi^T * softmax(beta * Xi * x + w) - x
    """
    z = (beta) * (Xi.T @ x) + w    # (K,)
    p = softmax(z)                      # (K,)
    return v + Xi @ p - x            # (N,) — Xi is (N,K), p is (K,)

In [30]:
from scipy.integrate import solve_ivp

def simulate(x0, Xi, beta, w, v, t_max=50, n_steps=500):
    """
    Simulate cell dynamics from an initial state.
    
    Parameters
    ----------
    x0     : np.array (N,) - starting cell state
    Xi     : np.array (N, K) - TF x cell type matrix
    beta   : float - commitment parameter
    w      : np.array (K,) - input field
    v      : np.array (N,) - overexpression vector
    t_max  : float - how long to simulate
    n_steps: int - number of timepoints

    Returns
    -------
    t    : timepoints
    traj : cell state over time (N x n_steps)
    """
    t_span = (0, t_max)
    t_eval = np.linspace(0, t_max, n_steps)

    sol = solve_ivp(
    lambda t, x: dx_dt(x, Xi, beta, w, v),
    t_span,
    x0,
    t_eval=t_eval,
    method='RK45',
    atol=1e-8,
    rtol=1e-8,
    max_step=0.01
)

    if not sol.success:
        raise RuntimeError(f"solve_ivp failed: {sol.message}")

    return sol.t, sol.y

In [31]:
def compute_v_overexpression(tf_name_list, overexpressed_tfs, delta=1.0):
    """
    Construct overexpression vector v.
    
    Parameters
    ----------
    tf_name_list      : list of TF names corresponding to rows of Xi
    overexpressed_tfs : list of TF names to overexpress
    delta             : overexpression magnitude (default 1.0)
    
    Returns
    -------
    v : np.array (N,) — delta at overexpressed TF indices, 0 elsewhere
    """
    v = np.zeros(len(tf_name_list))
    for tf in overexpressed_tfs:
        if tf in tf_name_list:
            v[tf_name_list.index(tf)] = delta
        else:
            print(f"Warning: {tf} not found in TF list")
    return v

In [32]:
def identify_cell_type(x_final, Xi_array, cell_type_names):
    """
    Find which cell type the final state is closest to.
    Uses cosine similarity.
    """
    similarities = {}
    for i, name in enumerate(cell_type_names):
        xi = Xi_array[:, i]
        sim = np.dot(x_final, xi) / (np.linalg.norm(x_final) * np.linalg.norm(xi) + 1e-10)
        similarities[name] = sim
    
    best = max(similarities, key=similarities.get)
    return best, similarities




In [33]:
def run_reprogramming(Xi_df, source_cell, target_cell, beta, delta=1.0, top_n=3, 
                      t_overexpress=20, t_max=50, n_steps=500):
    """
    Full reprogramming simulation pipeline.
    
    Parameters
    ----------
    Xi_df         : DataFrame (N TFs x K cell types) - normalised Xi matrix
    source_cell   : str - starting cell type
    target_cell   : str - target cell type
    beta          : float - commitment parameter
    delta         : float - overexpression magnitude
    top_n         : int - number of top TFs to overexpress
    t_overexpress : float - duration of overexpression
    t_max         : float - total simulation time
    n_steps       : int - number of timepoints

    Returns
    -------
    result_base   : str - identified cell type without overexpression
    result_reprog : str - identified cell type with overexpression
    similarities  : dict - cosine similarity scores for reprogrammed state
    top_tfs       : list - names of overexpressed TFs
    """
    Xi_array = Xi_df.values
    cell_type_names = Xi_df.columns.tolist()
    tf_name_list = Xi_df.index.tolist()

    # Step 1 — rank TFs by predictivity gap
    rankings = rank_tfs(Xi_df, source_cell, target_cell, top_n=top_n)
    top_tfs = rankings.index.tolist()

    print(f"Top {top_n} TFs for {source_cell} → {target_cell}:")
    for tf in top_tfs:
        print(f"  {tf}")

    # Step 2 — construct v vectors
    w = np.zeros(len(cell_type_names))
    v_zero = np.zeros(len(tf_name_list))
    v_reprog = compute_v_overexpression(tf_name_list, top_tfs, delta=delta)

    # Step 3 — initial condition
    x0 = Xi_array[:, cell_type_names.index(source_cell)].copy()

    # Step 4 — simulate baseline
    _, traj_base = simulate(x0, Xi_array, beta, w, v_zero, t_max=t_max, n_steps=n_steps)
    result_base, _ = identify_cell_type(traj_base[:, -1], Xi_array, cell_type_names)

    # Step 5 — phase 1: simulate WITH overexpression for t_overexpress
    _, traj_reprog = simulate(x0, Xi_array, beta, w, v_reprog,
                              t_max=t_overexpress, n_steps=n_steps//5)

    # Step 6 — phase 2: continue WITHOUT overexpression from where phase 1 ended
    x_mid = traj_reprog[:, -1]
    _, traj_free = simulate(x_mid, Xi_array, beta, w, v_zero,
                            t_max=t_max - t_overexpress, n_steps=n_steps)

    result_reprog, similarities = identify_cell_type(traj_free[:, -1], Xi_array, cell_type_names)

    # Step 7 — report
    print(f"\nWithout overexpression → {result_base}")
    print(f"With overexpression    → {result_reprog}")
    print(f"\nSimilarity scores:")
    for name, score in sorted(similarities.items(), key=lambda x: -x[1]):
        print(f"  {name:<25} {score:.3f}")

    return result_base, result_reprog, similarities, top_tfs

In [34]:

# Load and transpose to (TFs x cell types)
df = pd.read_csv('FACS_Xi_matrix.csv', index_col=0)
Xi_facs = df.T  


In [41]:
result_base, result_reprog, similarities, top_tfs = run_reprogramming(
    Xi_facs,
    source_cell='fibroblast',
    target_cell='oligodendrocyte',
    beta=100,
    delta=1.0,
    top_n=3,
    t_overexpress=20,
    t_max=50,
    n_steps=500
)

Top 3 TFs for fibroblast → oligodendrocyte:
  Zfp536
  Etv1
  Sox10


/var/folders/qy/dvtnfp454rz18nny_zfzgrqm0000gn/T/ipykernel_53280/1375105568.py:5: RuntimeWarning: divide by zero encountered in matmul
  z = (beta) * (Xi.T @ x) + w    # (K,)
/var/folders/qy/dvtnfp454rz18nny_zfzgrqm0000gn/T/ipykernel_53280/1375105568.py:5: RuntimeWarning: overflow encountered in matmul
  z = (beta) * (Xi.T @ x) + w    # (K,)
/var/folders/qy/dvtnfp454rz18nny_zfzgrqm0000gn/T/ipykernel_53280/1375105568.py:5: RuntimeWarning: invalid value encountered in matmul
  z = (beta) * (Xi.T @ x) + w    # (K,)
/var/folders/qy/dvtnfp454rz18nny_zfzgrqm0000gn/T/ipykernel_53280/1375105568.py:7: RuntimeWarning: divide by zero encountered in matmul
  return v + Xi @ p - x            # (N,) — Xi is (N,K), p is (K,)
/var/folders/qy/dvtnfp454rz18nny_zfzgrqm0000gn/T/ipykernel_53280/1375105568.py:7: RuntimeWarning: overflow encountered in matmul
  return v + Xi @ p - x            # (N,) — Xi is (N,K), p is (K,)
/var/folders/qy/dvtnfp454rz18nny_zfzgrqm0000gn/T/ipykernel_53280/1375105568.py:7: Ru


Without overexpression → fibroblast
With overexpression    → oligodendrocyte

Similarity scores:
  oligodendrocyte           1.000
  oligodendrocyte precursor cell 0.822
  neuronal stem cell        0.648
  Bergmann glial cell       0.636
  astrocyte of the cerebral cortex 0.596
  neuron                    0.544
  stromal cell              0.527
  pancreatic stellate cell  0.525
  mesenchymal stem cell     0.493
  mesenchymal stem cell of adipose 0.487
  mesenchymal cell          0.476
  fibroblast                0.458
  pancreatic PP cell        0.449
  pancreatic D cell         0.447
  basal cell                0.439
  epithelial cell           0.425
  pancreatic A cell         0.424
  pancreatic ductal cell    0.421
  brain pericyte            0.415
  skeletal muscle satellite cell 0.415
  smooth muscle cell        0.411
  mesothelial cell          0.407
  pancreatic acinar cell    0.407
  endothelial cell          0.403
  type I pneumocyte         0.390
  keratinocyte stem cell    